# DeepIO Parser Aggregator

## Generación de métricas promedio de E/S a partir de trazas Darshan Parser

Este notebook procesa los archivos CSV generados por Darshan Parser y calcula métricas agregadas de comportamiento de entrada/salida para aplicaciones de Inteligencia Artificial ejecutadas en sistemas HPC.

Las métricas obtenidas se utilizan para:

- Caracterización de patrones de E/S.
- Evaluación de rendimiento.
- Análisis de escalabilidad.
- Construcción de tablas para publicaciones científicas y tesis doctorales.

Autor: Edixon Parraga
Proyecto: Sintonización del sistema de E/S para aplicaciones de IA ejecutadas en sistemas HPC

# Objetivos

A partir de los registros generados por Darshan Parser:

1. Agrupar ejecuciones equivalentes.
2. Calcular métricas promedio.
3. Calcular desviaciones estándar.
4. Derivar indicadores de caracterización.
5. Derivar indicadores de metadatos.
6. Exportar resultados para análisis posteriores.

In [1]:
pwd

'F:\\NoteBooks\\PHD_THESIS\\NoteBooks\\DeepIO_Parser_HPC'

# Librerías

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Parámetros globales

In [3]:
# =====================================================
# Configuración
# =====================================================

INPUT_FILE = "Parser_DG_nfs.csv"

OUTPUT_DIR = Path("output_parser")
OUTPUT_DIR.mkdir(exist_ok=True)

# Tamaño físico del dataset HDF5 DeepGalaxy

DATASET_SIZE_BYTES = 2038916648

# Mapeo de modos

MODE_MAP = {
    0: "shared",
    1: "shared_reload_shuffle"
}

# Lectura de datos

Se carga el fichero CSV generado por Darshan Parser.

In [4]:
df = pd.read_csv(INPUT_FILE, sep=";")

print("Número de filas:", len(df))
print("Número de columnas:", len(df.columns))

df.head()

Número de filas: 26
Número de columnas: 92


,Jobid,File Format,File System Type,Process_IO,File Name,Access_Mode,POSIX_ACCESS1_ACCESS,POSIX_ACCESS1_COUNT,POSIX_ACCESS2_ACCESS,POSIX_ACCESS2_COUNT,POSIX_ACCESS3_ACCESS,POSIX_ACCESS3_COUNT,POSIX_ACCESS4_ACCESS,POSIX_ACCESS4_COUNT,POSIX_BYTES_READ,POSIX_BYTES_WRITTEN,POSIX_CONSEC_READS,POSIX_CONSEC_WRITES,POSIX_DUPS,POSIX_FASTEST_RANK,POSIX_FASTEST_RANK_BYTES,POSIX_FDSYNCS,POSIX_FILENOS,POSIX_FILE_ALIGNMENT,POSIX_FILE_NOT_ALIGNED,POSIX_FSYNCS,POSIX_F_CLOSE_END_TIMESTAMP,POSIX_F_CLOSE_START_TIMESTAMP,POSIX_F_FASTEST_RANK_TIME,POSIX_F_MAX_READ_TIME,POSIX_F_MAX_WRITE_TIME,POSIX_F_META_TIME,POSIX_F_OPEN_END_TIMESTAMP,POSIX_F_OPEN_START_TIMESTAMP,POSIX_F_READ_END_TIMESTAMP,POSIX_F_READ_START_TIMESTAMP,POSIX_F_READ_TIME,POSIX_F_SLOWEST_RANK_TIME,POSIX_F_VARIANCE_RANK_BYTES,POSIX_F_VARIANCE_RANK_TIME,POSIX_F_WRITE_END_TIMESTAMP,POSIX_F_WRITE_START_TIMESTAMP,POSIX_F_WRITE_TIME,POSIX_MAX_BYTE_READ,POSIX_MAX_BYTE_WRITTEN,POSIX_MAX_READ_TIME_SIZE,POSIX_MAX_WRITE_TIME_SIZE,POSIX_MEM_ALIGNMENT,POSIX_MEM_NOT_ALIGNED,POSIX_MMAPS,POSIX_MODE,POSIX_OPENS,POSIX_READS,POSIX_RENAMED_FROM,POSIX_RENAME_SOURCES,POSIX_RENAME_TARGETS,POSIX_RW_SWITCHES,POSIX_SEEKS,POSIX_SEQ_READS,POSIX_SEQ_WRITES,POSIX_SIZE_READ_0_100,POSIX_SIZE_READ_100K_1M,POSIX_SIZE_READ_100M_1G,POSIX_SIZE_READ_100_1K,POSIX_SIZE_READ_10K_100K,POSIX_SIZE_READ_10M_100M,POSIX_SIZE_READ_1G_PLUS,POSIX_SIZE_READ_1K_10K,POSIX_SIZE_READ_1M_4M,POSIX_SIZE_READ_4M_10M,POSIX_SIZE_WRITE_0_100,POSIX_SIZE_WRITE_100K_1M,POSIX_SIZE_WRITE_100M_1G,POSIX_SIZE_WRITE_100_1K,POSIX_SIZE_WRITE_10K_100K,POSIX_SIZE_WRITE_10M_100M,POSIX_SIZE_WRITE_1G_PLUS,POSIX_SIZE_WRITE_1K_10K,POSIX_SIZE_WRITE_1M_4M,POSIX_SIZE_WRITE_4M_10M,POSIX_SLOWEST_RANK,POSIX_SLOWEST_RANK_BYTES,POSIX_STATS,POSIX_STRIDE1_COUNT,POSIX_STRIDE1_STRIDE,POSIX_STRIDE2_COUNT,POSIX_STRIDE2_STRIDE,POSIX_STRIDE3_COUNT,POSIX_STRIDE3_STRIDE,POSIX_STRIDE4_COUNT,POSIX_STRIDE4_STRIDE,POSIX_WRITES
0,4942237.0,hdf5,nfs,4.0,output_bw_512,0.0,202.0,308304.0,3656.0,9072.0,512.0,4758.0,142.0,2016.0,2.041644e+09,0.0,467215.0,0.0,0.0,0.0,5.104133e+08,0.0,0.0,32768.0,500861.0,0.0,0.0,0.0,8.064991,0.168898,0.0,0.000653,0.015846,0.011546,227.363516,0.021873,34.860590,9.955236,3.970112e+06,0.541647,0.0,0.0,0.0,2.038917e+09,0.0,34482.0,0.0,8.0,0.0,-1.0,0.0,4.0,500886.0,0.0,0.0,0.0,0.0,0.0,488785.0,0.0,12.0,0.0,0.0,347586.0,71388.0,0.0,0.0,81900.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,5.104088e+08,8.0,576.0,1536.0,352.0,272.0,152.0,88.0,144.0,7887876.0,0.0
1,5175899.0,hdf5,nfs,4.0,output_bw_512,0.0,202.0,308304.0,3656.0,9072.0,512.0,4758.0,142.0,2016.0,2.041644e+09,0.0,467215.0,0.0,0.0,3.0,5.104088e+08,0.0,0.0,32768.0,500861.0,0.0,0.0,0.0,5.377684,0.173053,0.0,0.001429,0.011030,0.007554,205.880054,0.009189,25.369716,7.340848,3.970112e+06,0.521643,0.0,0.0,0.0,2.038917e+09,0.0,29993.0,0.0,8.0,0.0,-1.0,0.0,4.0,500886.0,0.0,0.0,0.0,0.0,0.0,488785.0,0.0,12.0,0.0,0.0,347586.0,71388.0,0.0,0.0,81900.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,5.104128e+08,8.0,576.0,1536.0,352.0,272.0,152.0,88.0,144.0,7887876.0,0.0
2,5186375.0,hdf5,nfs,4.0,output_bw_512,0.0,202.0,2466432.0,3656.0,72576.0,512.0,8800.0,328.0,2516.0,1.631330e+10,0.0,3734416.0,0.0,0.0,3.0,4.078325e+09,0.0,0.0,32768.0,3956488.0,0.0,0.0,0.0,3.182102,0.081074,0.0,0.001657,0.013306,0.010009,32735.495060,0.012479,41.875229,19.952294,0.000000e+00,43.290731,0.0,0.0,0.0,2.038917e+09,0.0,3656.0,0.0,8.0,0.0,-1.0,0.0,4.0,3956632.0,0.0,0.0,0.0,0.0,0.0,3880040.0,0.0,12.0,0.0,0.0,2730316.0,571104.0,0.0,0.0,655200.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.078325e+09,8.0,1280.0,272.0,576.0,1536.0,152.0,88.0,144.0,7887876.0,0.0
3,5124492.0,hdf5,nfs,8.0,output_bw_512,0.0,202.0,4894622.0,3656.0,144027.0,512.0,17531.0,328.0,5008.0,3.237276e+10,0.0,7410809.0,0.0,0.0,7.0,4.015581e+09,0.0,0.0,32768.0,7851616.0,0.0,0.0,0.0,6.219543,0.843553,0.0,0.141407,0.095627,0.069708,15121.638650,0.089601,94.938188,20.355897,9.360000e+14,22.702253,0.0,0.0,0.0,2.038917e+09,0.0,5933.0,0.0,8.0,0.0,-1.0,0.0,8.0,785

# Limpieza y normalización

In [5]:
df.columns = df.columns.str.strip()

numeric_columns = [
    "Jobid",
    "Process_IO",
    "Access_Mode",
    "POSIX_READS",
    "POSIX_BYTES_READ",
    "POSIX_SEQ_READS",
    "POSIX_CONSEC_READS",
    "POSIX_F_META_TIME",
    "POSIX_F_READ_TIME"
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Cálculo de métricas por ejecución

In [6]:
df["Sequential_Reads_pct"] = (
    df["POSIX_SEQ_READS"] /
    df["POSIX_READS"]
) * 100

df["Consecutive_Reads_pct"] = (
    df["POSIX_CONSEC_READS"] /
    df["POSIX_READS"]
) * 100

df["Avg_Read_Size_bytes"] = (
    df["POSIX_BYTES_READ"] /
    df["POSIX_READS"]
)

df["MII_pct"] = (
    df["POSIX_F_META_TIME"] /
    df["POSIX_F_READ_TIME"]
) * 100

df["Access_Mode_Label"] = (
    df["Access_Mode"]
    .map(MODE_MAP)
)

# Agrupación de ejecuciones equivalentes

Las ejecuciones se agrupan por:

- Modo de acceso
- Número de procesos

In [7]:
group_cols = [
    "Access_Mode",
    "Access_Mode_Label",
    "Process_IO"
]

# Promedios y desviaciones estándar

In [8]:
agg = (
    df
    .groupby(group_cols)
    .agg(
        n_runs=("Jobid","count"),

        POSIX_READS_mean=("POSIX_READS","mean"),
        POSIX_READS_std=("POSIX_READS","std"),

        POSIX_BYTES_READ_mean=("POSIX_BYTES_READ","mean"),
        POSIX_BYTES_READ_std=("POSIX_BYTES_READ","std"),

        Sequential_Reads_pct_mean=("Sequential_Reads_pct","mean"),

        Consecutive_Reads_pct_mean=("Consecutive_Reads_pct","mean"),

        Avg_Read_Size_bytes_mean=("Avg_Read_Size_bytes","mean"),

        POSIX_F_META_TIME_mean=("POSIX_F_META_TIME","mean"),

        POSIX_F_READ_TIME_mean=("POSIX_F_READ_TIME","mean"),

        MII_pct_mean=("MII_pct","mean")
    )
    .reset_index()
)

# Métricas derivadas

Se calculan:

- rho_rel
- rho_phys
- variaciones relativas

In [9]:
agg["rho_phys"] = (
    agg["POSIX_BYTES_READ_mean"]
    /
    DATASET_SIZE_BYTES
)

agg["rho_rel"] = np.nan

In [10]:
for mode in agg["Access_Mode"].unique():

    mask = agg["Access_Mode"] == mode

    base = agg.loc[
        mask &
        (agg["Process_IO"] == 4),
        "POSIX_BYTES_READ_mean"
    ]

    if len(base) > 0:

        base_value = base.iloc[0]

        agg.loc[mask, "rho_rel"] = (
            agg.loc[mask, "POSIX_BYTES_READ_mean"]
            /
            base_value
        )

# Variaciones relativas

In [11]:
metrics = [
    "POSIX_READS_mean",
    "POSIX_BYTES_READ_mean",
    "Avg_Read_Size_bytes_mean",
    "POSIX_F_META_TIME_mean",
    "MII_pct_mean"
]

for metric in metrics:

    agg[f"Delta_{metric}"] = np.nan

for mode in agg["Access_Mode"].unique():

    mask = agg["Access_Mode"] == mode

    base_row = agg[
        mask &
        (agg["Process_IO"] == 4)
    ]

    if len(base_row) == 0:
        continue

    for metric in metrics:

        base_value = base_row[metric].iloc[0]

        agg.loc[
            mask,
            f"Delta_{metric}"
        ] = (
            (
                agg.loc[mask, metric]
                -
                base_value
            )
            /
            base_value
        ) * 100

# Tabla agregada final

In [12]:
agg

,Access_Mode,Access_Mode_Label,Process_IO,n_runs,POSIX_READS_mean,POSIX_READS_std,POSIX_BYTES_READ_mean,POSIX_BYTES_READ_std,Sequential_Reads_pct_mean,Consecutive_Reads_pct_mean,Avg_Read_Size_bytes_mean,POSIX_F_META_TIME_mean,POSIX_F_READ_TIME_mean,MII_pct_mean,rho_phys,rho_rel,Delta_POSIX_READS_mean,Delta_POSIX_BYTES_READ_mean,Delta_Avg_Read_Size_bytes_mean,Delta_POSIX_F_META_TIME_mean,Delta_MII_pct_mean
0,0.0,shared,4.0,3,1.652801e+06,1.995176e+06,6.798862e+09,8.239743e+09,97.744125,93.646377,4091.719308,0.001246,34.035178,0.003821,3.334547,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.0,shared,8.0,3,2.956037e+06,4.239945e+06,1.215391e+10,1.751003e+10,97.386704,92.824457,4056.809768,0.048660,60.721840,0.053176,5.960964,1.787639,78.850130,78.763870,-0.853175,3804.252474,1291.684934
2,0.0,shared,16.0,3,5.390795e+06,8.406572e+06,2.214996e+10,3.471234e+10,96.736486,91.334811,3990.694226,0.005991,74.631700,0.013249,10.863592,3.257892,226.161079,225.789198,-2.469013,380.716769,246.755808
3,0.0,shared,32.0,1,2.804532e+07,NaN,1.160000e+11,NaN,98.054970,94.357265,4136.162616,0.017303,199.315584,0.008681,56.892958,17.061678,1596.835454,1606.167791,1.086177,1288.312383,127.199875
4,0.0,shared,48.0,2,1.838080e+07,2.510664e+07,7.562847e+10,1.037630e+11,95.376930,88.258207,3859.482294,0.023981,217.553124,0.014509,37.092476,11.123694,1012.099902,1012.369426,-5.675781,1824.164215,279.714150
5,0.0,shared,64.0,2,6.729900e+05,0.000000e+00,2.340588e+09,0.000000e+00,91.366291,79.146050,3477.893908,0.025021,141.431844,0.018939,1.147957,0.344262,-59.281858,-65.573831,-15.001650,1907.608986,395.663623
6,1.0,shared_reload_shuffle,4.0,2,3.950268e+06,6.915504e+02,1.628691e+10,2.029812e+06,98.064055,94.383381,4122.989895,0.001625,33.802619,0.004799,7.988024,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,1.0,shared_reload_shuffle,8.0,2,4.197696e+06,5.217872e+06,1.728176e+10,2.154876e+10,97.556057,93.214251,4073.361107,0.002933,50.569099,0.005876,8.475951,1.061082,6.263588,6.108231,-1.203709,80.467548,22.449481
8,1.0,shared_reload_shuffle,16.0,2,1.517585e+07,6.031211e+05,6.257330e+10,2.497590e+09,98.061113,94.375938,4123.200609,0.005425,102.598961,0.005306,30.689485,3.841937,284.172745,284.193698,0.005111,233.712704,10.576896
9,1.0,shared_reload_shuffle,32.0,3,5.824620e+05,0.000000e+00,2.187401e+09,0.000000e+00,94.265892,85.713231,3755.440247,0.019151,73.628823,0.030410,1.072825,0.134304,-85.255127,-86.569579,-8.914639,1078.181072,533.717707


# Casos extremos utilizados en la tesis

In [13]:
summary_extremos = agg[
    agg["Process_IO"].isin([4,64])
].copy()

summary_extremos

,Access_Mode,Access_Mode_Label,Process_IO,n_runs,POSIX_READS_mean,POSIX_READS_std,POSIX_BYTES_READ_mean,POSIX_BYTES_READ_std,Sequential_Reads_pct_mean,Consecutive_Reads_pct_mean,Avg_Read_Size_bytes_mean,POSIX_F_META_TIME_mean,POSIX_F_READ_TIME_mean,MII_pct_mean,rho_phys,rho_rel,Delta_POSIX_READS_mean,Delta_POSIX_BYTES_READ_mean,Delta_Avg_Read_Size_bytes_mean,Delta_POSIX_F_META_TIME_mean,Delta_MII_pct_mean
0,0.0,shared,4.0,3,1.652801e+06,1.995176e+06,6.798862e+09,8.239743e+09,97.744125,93.646377,4091.719308,0.001246,34.035178,0.003821,3.334547,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,0.0,shared,64.0,2,6.729900e+05,0.000000e+00,2.340588e+09,0.000000e+00,91.366291,79.146050,3477.893908,0.025021,141.431844,0.018939,1.147957,0.344262,-59.281858,-65.573831,-15.001650,1907.608986,395.663623
6,1.0,shared_reload_shuffle,4.0,2,3.950268e+06,6.915504e+02,1.628691e+10,2.029812e+06,98.064055,94.383381,4122.989895,0.001625,33.802619,0.004799,7.988024,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
11,1.0,shared_reload_shuffle,64.0,1,4.092304e+07,NaN,1.690000e+11,NaN,98.023253,94.272798,4129.702974,0.015838,283.952944,0.005578,82.887155,10.376428,935.956041,937.642796,0.162821,874.346355,16.233724


# Exportación de resultados

In [14]:
agg.to_csv(
    OUTPUT_DIR /
    "Parser_DG_nfs_promedios.csv",
    index=False
)

summary_extremos.to_csv(
    OUTPUT_DIR /
    "Parser_DG_nfs_promedios_4p_64p.csv",
    index=False
)

print("Archivos exportados correctamente")

Archivos exportados correctamente


# Exportación a LaTeX

In [15]:
latex_table = summary_extremos.to_latex(
    index=False,
    float_format="%.4f"
)

with open(
    OUTPUT_DIR /
    "tabla_patron_nfs.tex",
    "w",
    encoding="utf-8"
) as f:

    f.write(latex_table)

print("Tabla LaTeX exportada")

Tabla LaTeX exportada
